In [6]:
# ==========================================================
# 1. IMPORTING TOOLS (LIBRARIES)
# ==========================================================

# 'os' is a tool that lets Python talk to your computer's operating system.
# We use it here to find 'environment variables', which are like secret settings.
import os

# 'pandas' is our table-handling tool, as we saw in the first notebook.
import pandas as pd

# 'dotenv' helps us read a special file called '.env' where we keep secret keys (like API keys).
# This keeps our secrets safe and out of our main code.
from dotenv import load_dotenv

# ==========================================================
# 2. LOADING OUR SECRET KEYS
# ==========================================================

# This line looks for the '.env' file and loads the secrets inside it into Python's memory.
load_dotenv('../.env')

# ==========================================================
# 3. CHECKING THE API KEY
# ==========================================================

# We ask 'os' to get the secret key named 'GROQ_API_KEY'.
api_key = os.getenv("GROQ_API_KEY")

# If the key was found, we print a success message showing just the start of the key for safety.
# If not, we print an error message so we know something went wrong.
if api_key:
    print("API key loaded successfully:", api_key[:10], "...")
else:
    print("ERROR: API key not found. Check your .env file path.")

# ==========================================================
# 4. SETTING UP THE AI (LANGCHAIN + GROQ)
# ==========================================================

# 'ChatGroq' is the brain of our operation; it connects us to a powerful AI model.
from langchain_groq import ChatGroq

# 'PromptTemplate' is like a fill-in-the-blanks form we use to talk to the AI.
from langchain_core.prompts import PromptTemplate

# 'JsonOutputParser' ensures the AI gives us answers in a structured format (JSON) that Python can easily read.
from langchain_core.output_parsers import JsonOutputParser

API key loaded successfully: gsk_ju8IrY ...


In [7]:
# Now we load the data we cleaned in the first notebook. 
# Remember, we saved it in the 'processed' folder as 'returns_clean.csv'.
df = pd.read_csv("../data/processed/returns_clean.csv")

# We use .head() again just to make sure the data loaded correctly and everything is in its place.
df.head()

,Order_ID,Product_ID,User_ID,Order_Date,Product_Category,Product_Price,Order_Quantity,Discount_Applied,Shipping_Method,Payment_Method,...,Return_Status,Return_Reason,Days_to_Return,Order_Value,Return_Cost,Profit_Loss,CO2_Emissions,Packaging_Waste,CO2_Saved,Waste_Avoided
0,ORD00000,PROD0169,USER0195,2022-01-14,Clothing,1720.71,2,30.46,Next-Day,Wallet,...,Not Returned,No Return,0,2393.163468,0,2393.163468,2.0,0.4,2.0,0.4
1,ORD00001,PROD0318,USER1469,2022-01-03,Toys,744.06,5,29.62,Next-Day,Wallet,...,Returned,Size Issue,12,2618.347140,200,2418.347140,2.0,1.0,0.0,0.0
2,ORD00002,PROD0427,USER1812,2025-03-16,Clothing,983.68,5,47.80,Express,Wallet,...,Not Returned,No Return,0,2567.404800,0,2567.404800,1.5,1.0,1.5,1.0
3,ORD00003,PROD0323,USER1274,2024-11-06,Books,1855.65,2,2.90,Express,COD,...,Not Returned,No Return,0,3603.672300,0,3603.672300,1.5,0.4,1.5,0.4
4,ORD00004,PROD0325,USER0551,2023-06-07,Home Appliances,1770.97,5,44.42,Express,COD,...,Returned,Size Issue,11,4921.525630,200,4721.525630,1.5,1.0,0.0,0.0


In [8]:
# Here we initialize the AI model. 
# We tell it which model to use ('llama-3.1-8b-instant'), give it our API key, and set 'temperature=0'.
# Temperature=0 makes the AI's answers very consistent and less 'creative', which is good for sorting data.
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",  
    api_key=api_key,
    temperature=0
)

# To make sure the AI is listening, we send it a simple test message: "Say hello in one line".
response = llm.invoke([
    {"role": "user", "content": "Say hello in one line"}
])

# We print the AI's reply to verify everything is working.
print(response.content)

Hello, how are you today?


In [9]:
# We define a list of categories that we want the AI to use when sorting the return reasons.
# These are the only options the AI is allowed to pick from.
CATEGORIES = [
    "SIZE_FIT_ISSUE",
    "DESCRIPTION_MISMATCH",
    "QUALITY_DEFECT",
    "WRONG_ITEM_SENT",
    "CHANGED_MIND",
    "DAMAGED_IN_SHIPPING",
    "COUNTERFEIT_SUSPECTED"
]

In [10]:
# We create a 'parser' that will take the AI's text response and turn it into a Python dictionary.
# This makes it easy for our code to use the information the AI gives us.
parser = JsonOutputParser()

In [11]:
# This is the 'PromptTemplate'. It's a set of instructions we send to the AI every time.
# We tell the AI what its job is, give it the list of categories, and set strict rules for how it should answer.
# The curly braces {return_reason} and {categories} are placeholders that will be filled with real data later.
prompt = PromptTemplate(
    template="""
You are an AI system that classifies e-commerce return reasons.

You MUST choose ONLY ONE category from this list:
{categories}

STRICT RULES:
- Output must be valid JSON
- Do not add explanations outside JSON
- Category must be EXACTLY from list
- Confidence must be between 0 and 1

Return format:
{{
    "category": "...",
    "confidence": 0.0,
    "key_signal": "short phrase"
}}

Return reason:
"{return_reason}"
""",
    input_variables=["return_reason"],
    partial_variables={"categories": CATEGORIES}
)

In [12]:
# We combine our instructions (prompt), the AI (llm), and the parser into a single 'chain'.
# The '|' symbol means 'take the output from the left and pass it to the right'.
# So: instructions go to the AI, and the AI's answer goes to the parser.
chain = prompt | llm | parser

In [13]:
# We test our 'chain' with a single example to see if it works as expected.
# We provide a sample return reason and wait for the AI to categorize it.
test_text = "The size was too small and color looked different"

# .invoke() runs the entire chain and gives us the final result.
result = chain.invoke({"return_reason": test_text})

# We print the result to see the category, confidence level, and the key signal identified by the AI.
print(result)

{'category': 'SIZE_FIT_ISSUE', 'confidence': 0.8, 'key_signal': 'size issue'}


In [14]:
# We create a function called 'classify_safe' that handles errors gracefully.
# If the AI gets confused or the internet cuts out, the 'try-except' block will catch the error.
# Instead of the whole program crashing, it will just return an 'ERROR' message for that specific line.
def classify_safe(text):
    try:
        return chain.invoke({"return_reason": text})
    
    except Exception as e:
        return {
            "category": "ERROR",
            "confidence": 0,
            "key_signal": str(e)
        }

In [15]:
# We create a list of sample return reasons to test our 'classify_safe' function.
samples = [
    "Product quality was very poor",
    "Received wrong item",
    "Changed my mind after ordering",
    "Package was damaged"
]

# We use a 'for' loop to go through each sample, categorize it, and print the result.
for s in samples:
    print("Input:", s)
    print("Output:", classify_safe(s))
    print("-" * 50)

Input: Product quality was very poor
Output: {'category': 'QUALITY_DEFECT', 'confidence': 0.9, 'key_signal': 'poor quality'}
--------------------------------------------------
Input: Received wrong item
Output: {'category': 'WRONG_ITEM_SENT', 'confidence': 1.0, 'key_signal': 'wrong item'}
--------------------------------------------------
Input: Changed my mind after ordering
Output: {'category': 'CHANGED_MIND', 'confidence': 1.0, 'key_signal': 'changed mind'}
--------------------------------------------------
Input: Package was damaged
Output: {'category': 'DAMAGED_IN_SHIPPING', 'confidence': 1.0, 'key_signal': 'Package was damaged'}
--------------------------------------------------


In [18]:
# Processing items one by one can be slow. Here we create a 'batch' function.
# This function sends several reasons to the AI at once, which is much faster.
# It asks the AI to return a list of JSON objects, one for each reason in the batch.
def classify_batch(text_list):
    try:
        prompt_text = f"""
Classify each return reason into one of these categories:
{CATEGORIES}

Return STRICT JSON as a list:
[
  {{"category": "...", "confidence": 0.0, "key_signal": "..."}}
]

Return reasons:
{text_list}
"""

        # We call the AI directly with the batch prompt.
        response = llm.invoke(prompt_text)
        
        # We use the 'json' library to turn the AI's text response back into a Python list.
        import json
        return json.loads(response.content)
    
    except Exception as e:
        # If the batch processing fails, we mark all items in that batch as 'ERROR'.
        return [{"category": "ERROR", "confidence": 0, "key_signal": str(e)}] * len(text_list)

In [19]:
# 'tqdm' is a library that adds a progress bar to our loops so we can see how much work is left.
from tqdm import tqdm
 
# We process our data in chunks of 10 items at a time.
batch_size = 10
results = []

# We take the 'Return_Reason' column from our table and turn it into a list.
# .fillna("") ensures that if any reason is missing, it's replaced with an empty text instead of causing an error.
texts = df['Return_Reason'].fillna("").tolist()

# This loop goes through the list 10 items at a time, categorizes them, and adds them to our 'results' list.
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    batch_result = classify_batch(batch)
    results.extend(batch_result)

100%|██████████| 500/500 [1:06:08<00:00,  7.94s/it]


In [20]:
# We turn our list of results into a new table (results_df).
results_df = pd.DataFrame(results)

# We then glue this new table to the side of our original table (df) using 'pd.concat'.
# Now each order has its AI-assigned category right next to it!
df = pd.concat([df, results_df], axis=1)

In [21]:
# We peek at the first 5 rows of our updated table to see the new columns in action.
df.head()

,Order_ID,Product_ID,User_ID,Order_Date,Product_Category,Product_Price,Order_Quantity,Discount_Applied,Shipping_Method,Payment_Method,...,Order_Value,Return_Cost,Profit_Loss,CO2_Emissions,Packaging_Waste,CO2_Saved,Waste_Avoided,category,confidence,key_signal
0,ORD00000,PROD0169,USER0195,2022-01-14,Clothing,1720.71,2,30.46,Next-Day,Wallet,...,2393.163468,0,2393.163468,2.0,0.4,2.0,0.4,ERROR,0,Expecting value: line 1 column 1 (char 0)
1,ORD00001,PROD0318,USER1469,2022-01-03,Toys,744.06,5,29.62,Next-Day,Wallet,...,2618.347140,200,2418.347140,2.0,1.0,0.0,0.0,ERROR,0,Expecting value: line 1 column 1 (char 0)
2,ORD00002,PROD0427,USER1812,2025-03-16,Clothing,983.68,5,47.80,Express,Wallet,...,2567.404800,0,2567.404800,1.5,1.0,1.5,1.0,ERROR,0,Expecting value: line 1 column 1 (char 0)
3,ORD00003,PROD0323,USER1274,2024-11-06,Books,1855.65,2,2.90,Express,COD,...,3603.672300,0,3603.672300,1.5,0.4,1.5,0.4,ERROR,0,Expecting value: line 1 column 1 (char 0)
4,ORD00004,PROD0325,USER0551,2023-06-07,Home Appliances,1770.97,5,44.42,Express,COD,...,4921.525630,200,4721.525630,1.5,1.0,0.0,0.0,ERROR,0,Expecting value: line 1 column 1 (char 0)


In [27]:
# We redefine our 'classify_safe' function here just to be absolutely sure we have the latest version.
def classify_safe(text):
    try:
        return chain.invoke({"return_reason": text})
    except Exception as e:
        return {
            "category": "ERROR",
            "confidence": 0,
            "key_signal": str(e)
        }

In [28]:
# We test the function on the first 3 return reasons in our table to double-check the results.
for text in df['Return_Reason'].head(3):
    print(classify_safe(text))

{'category': 'CHANGED_MIND', 'confidence': 1.0, 'key_signal': 'No Return'}
{'category': 'SIZE_FIT_ISSUE', 'confidence': 1.0, 'key_signal': 'Size'}
{'category': 'CHANGED_MIND', 'confidence': 1.0, 'key_signal': 'No Return'}


In [29]:
# And one last test on the first 5 reasons, just to be thorough!
for text in df['Return_Reason'].head(5):
    print(classify_safe(text))

{'category': 'CHANGED_MIND', 'confidence': 1.0, 'key_signal': 'No Return'}
{'category': 'SIZE_FIT_ISSUE', 'confidence': 1.0, 'key_signal': 'Size'}
{'category': 'CHANGED_MIND', 'confidence': 1.0, 'key_signal': 'No Return'}
{'category': 'CHANGED_MIND', 'confidence': 1.0, 'key_signal': 'No Return'}
{'category': 'SIZE_FIT_ISSUE', 'confidence': 1.0, 'key_signal': 'Size'}


In [32]:
# We save our newly classified data into a new CSV file called 'classified_returns.csv'.
# This file now contains everything from the start, plus the AI's categories!
df.to_csv("../data/processed/classified_returns.csv", index=False)